# 10.6 · Exercises

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/10-deployment-mlops/06-monitoring-drift/exercises.ipynb)

In [ ]:
# --- course files: found locally, downloaded on Colab -------------------------
import sys, urllib.request
from pathlib import Path
REPO_RAW = "https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/"

def repo_file(rel):
    """Return a local path to a file in this repo, downloading it if needed."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / rel).exists():
            return parent / rel
    dest = Path("course_files") / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    if not dest.exists():
        urllib.request.urlretrieve(REPO_RAW + rel, dest)
    return dest

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
repo_file("data/download.py")                                  # churn_model finds the data through it
sys.path.insert(0, str(repo_file("lib/churn_model.py").parent))
import churn_model as cm

bundle = cm.load_bundle()          # trains and saves models/churn_bundle.joblib the first time
ROOT = cm.repo_root()
print(f"Model trained {bundle['trained']}, test AUC {bundle['metrics']['test_auc']}, "
      f"threshold {bundle['threshold']}")

## Exercise 1: PSI by hand

Reference bins: 30%, 30%, 20%, 10%, 10%. This month: 20%, 25%, 25%, 15%, 15%. Compute the PSI on paper, then check. Which band is it in?

In [ ]:
# your code here

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

sales = cm.load_sales(both_years=True)         # Dec 2009 - Dec 2011
HISTORY_DAYS = 274                              # same window as the production model (Dec 2010 - Aug 2011)
CUTOFFS = pd.date_range("2010-09-01", "2011-09-01", freq="MS")
tables = {c: cm.build_customer_table(sales, c, lookback_days=HISTORY_DAYS) for c in CUTOFFS}
print(f"{len(tables)} monthly customer tables, {CUTOFFS[0].date()} to {CUTOFFS[-1].date()}")

def new_model():
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))

## Exercise 2: a calibration alert

Monitoring the inputs alone missed the most important change. Write `calibration_alert(predicted, actual, tolerance=0.10)` and apply it month by month to the September 2010 model. In which months would it fire, and how many months later would you have known (labels take 91 days)?

In [ ]:
# your code here

## Exercise 3: rolling retraining

Simulate the policy "on the first of each month, retrain on the newest cut-off whose labels are complete (three months earlier)" for June to September 2011. Compare its AUC and calibration gap each month with the fixed September 2010 model.

In [ ]:
# your code here